# Assignment 1: MLP Classifier

In this notebook, you will implement a simple MLP classifier to perform Fashion MNIST image classification.

The neural network will use the following model:

- Input: $x\in\mathbb{R}^{28\times 28}$.
- First the model will flatten $x$ into a single vector in $\mathbb{R}^{784}$.
- Then, $h = \mathrm{ReLU}(W_1\cdot x) + b_1) \in\mathbb{R}^{256}$ is the hidden layer.
- The logits are given by $y = W_2\cdot h + b_2\in\mathbb{R}^{10}$.

In [ ]:
! pip install -q git+https://github.com/ontariotechu-csci4052u/csci4052u
import csci4052u
print("CSCI 4052U version:", csci4052u.__version__)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 46.4 MB/s eta 0:00:00
CSCI 4052U version: 2026-03-19


## Dataset

In [ ]:
# @title Load the Lightning Data Module.
frac_str = "0.1" # @param ["0.1","0.2","1.0"]
fraction = float(frac_str)

data = csci4052u.data.FashionMNISTDataModule(fraction=fraction)
data.prepare_data()

100%|██████████| 26.4M/26.4M [00:01<00:00, 14.4MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 231kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 4.29MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 19.4MB/s]


## Lightning Module

In [ ]:
from lightning import LightningModule
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLP(LightningModule):
  def __init__(self, lr=1e-3):
    super().__init__()
    self.save_hyperparameters()

    # Model architecture layers
    self.flatten = nn.Flatten()
    self.fc1 = nn.Linear(784, 256)
    self.fc2 = nn.Linear(256, 10)

  def forward(self, x):
    x = self.flatten(x)
    h = F.relu(self.fc1(x))
    logits = self.fc2(h)
    return logits

  def training_step(self, batch, batch_idx):
    x, y = batch
    logits = self(x)
    loss = F.cross_entropy(logits, y)

    # Calculate simple accuracy
    preds = torch.argmax(logits, dim=1)
    acc = (preds == y).float().mean()

    self.log("train_loss", loss, on_epoch=True, prog_bar=True)
    self.log("train_acc", acc, on_epoch=True, prog_bar=True)
    return loss

  def validation_step(self, batch, batch_idx):
    x, y = batch
    logits = self(x)
    loss = F.cross_entropy(logits, y)

    preds = torch.argmax(logits, dim=1)
    acc = (preds == y).float().mean()

    self.log("val_loss", loss, on_epoch=True, prog_bar=True)
    self.log("val_acc", acc, on_epoch=True, prog_bar=True)

  def test_step(self, batch, batch_idx):
    x, y = batch
    logits = self(x)
    loss = F.cross_entropy(logits, y)

    preds = torch.argmax(logits, dim=1)
    acc = (preds == y).float().mean()

    self.log("test_loss", loss, on_epoch=True, prog_bar=True)
    self.log("test_acc", acc, on_epoch=True, prog_bar=True)

  def configure_optimizers(self):
    return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)


## Training

In [ ]:
csci4052u.training.train(
    model_name="MLP",
    model=MLP(),
    datamodule=data,
    max_epochs=10,
    monitor_metric="val_loss",
)

INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ flatten │ Flatten │      0 │ train │     0 │
│ 1 │ fc1     │ Linear  │  200 K │ train │     0 │
│ 2 │ fc2     │ Linear  │  2.6 K │ train │     0 │
└───┴─────────┴─────────┴────────┴───────┴───────┘

Trainable params: 203 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 203 K                                                                                                
Total estimated model params size (MB): 0.814                                                                      
Modules in train mode: 3                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.8784999847412109     │
│         test_loss         │    0.36628082394599915    │
└───────────────────────────┴───────────────────────────┘

In [ ]:
csci4052u.report.summarize(all_versions=True)

                                       Training Summary                                        
┏━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┓
┃ Model ┃ Version   ┃ Train Acc ┃ Val Acc ┃ Test Acc ┃ Trainable ┃ Frozen ┃ Epochs ┃ Duration ┃
┡━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━┩
│ MLP   │ version_0 │    0.9148 │  0.8830 │   0.8785 │    203.5K │      0 │     10 │    6m 8s │
└───────┴───────────┴───────────┴─────────┴──────────┴───────────┴────────┴────────┴──────────┘